# Power Plants Task

In [83]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

## 1. Explore raw data

Inspect each source file to understand its structure, spot missing/duplicate values and data quality issues.


In [85]:
def explore_csv(file_path: str):
    df = pd.read_csv(file_path)
    print(f"FILE: {file_path}")
    print(f"Shape: {df.shape}")
    print(f"\nColumn dtypes:\n{df.dtypes}")
    print(f"\nNull counts:\n{df.isnull().sum()}")
    print(f"\nDuplicate rows: {df.duplicated().sum()}")
    print(f"\nFirst 3 rows:")
    display(df.head(3))
    print(f"\nBasic stats:")
    display(df.describe(include='all'))
    return df

gas_fr_raw = explore_csv('gas_fr_plants.csv')
gas_raw    = explore_csv('gas_plants.csv')
wind_raw   = explore_csv('wind_plants.csv')

FILE: gas_fr_plants.csv
Shape: (962, 5)

Column dtypes:
Date           object
Country        object
Technology     object
SiteName       object
Volume        float64
dtype: object

Null counts:
Date          481
Country       481
Technology    481
SiteName      481
Volume        487
dtype: int64

Duplicate rows: 480

First 3 rows:


,Date,Country,Technology,SiteName,Volume
0,01/01/2024,FR,Gas,Blenod-5,6753.0
1,02/01/2024,FR,Gas,Blenod-5,3896.0
2,03/01/2024,FR,Gas,Blenod-5,3636.0



Basic stats:


,Date,Country,Technology,SiteName,Volume
count,481,481,481,481,475.000000
unique,481,1,1,1,NaN
top,01/01/2024,FR,Gas,Blenod-5,NaN
freq,1,481,481,481,NaN
mean,NaN,NaN,NaN,NaN,5009.648421
std,NaN,NaN,NaN,NaN,1151.503508
min,NaN,NaN,NaN,NaN,3059.000000
25%,NaN,NaN,NaN,NaN,4078.500000
50%,NaN,NaN,NaN,NaN,5051.000000
75%,NaN,NaN,NaN,NaN,5929.500000


FILE: gas_plants.csv
Shape: (962, 5)

Column dtypes:
Date          object
Country       object
Technology    object
SiteName      object
Volume        object
dtype: object

Null counts:
Date          0
Country       0
Technology    0
SiteName      0
Volume        3
dtype: int64

Duplicate rows: 0

First 3 rows:


...

,Date,Country,Technology,SiteName,Volume
count,962,962,962,962,959
unique,481,1,1,2,859
top,01/01/2024,GB,Gas,Pembroke-1,5647
freq,2,962,962,481,3


FILE: wind_plants.csv
Shape: (957, 5)

Column dtypes:
Date           object
Country        object
Technology     object
SiteName       object
Volume        float64
dtype: object

Null counts:
Date          0
Country       0
Technology    0
SiteName      0
Volume        0
dtype: int64

Duplicate rows: 0

First 3 rows:


,Date,Country,Technology,SiteName,Volume
0,01/01/2024,GB,Wind,Hornsea-1,260.166079
1,02/01/2024,GB,Wind,Hornsea-1,709.480820
2,03/01/2024,GB,Wind,Hornsea-1,431.527680



Basic stats:


,Date,Country,Technology,SiteName,Volume
count,957,957,957,957,957.000000
unique,481,1,1,2,NaN
top,01/01/2024,GB,Wind,Hornsea-1,NaN
freq,2,957,957,481,NaN
mean,NaN,NaN,NaN,NaN,654.104050
std,NaN,NaN,NaN,NaN,4058.265477
min,NaN,NaN,NaN,NaN,-550.307927
25%,NaN,NaN,NaN,NaN,247.140967
50%,NaN,NaN,NaN,NaN,486.007385
75%,NaN,NaN,NaN,NaN,736.814133


## 2: Data cleaning decisions
Based on the exploration above, the cleaning strategy is:

| Issue | Action |
|---|---|
| Missing `Volume` | Fill with `0` |
| Non-numeric `Volume` | Coerce to NaN then fill with `0` |
| Missing `date` | Drop the row|
| Invalid `date` format | Coerce to NaT then drop |
| Duplicate `(date, SiteName)` | Keep row with the latest `updatetime` |
| Country as 2-letter code | Map to full country name for database storage |
| Missing `updatedby` | Fill with `'petroineos'` |
| Missing `updatetime` | Fill with current timestamp |

The `Technology` column is inferred from the source file name and is not present in the raw CSVs.

## 3. Implementation


In [88]:
class PowerPlants(object):

    def __init__(self):
        self.database_file = 'database.csv'

    def analyse_plant_data(self, file_path: str):
        """
        Cleaning steps applied:
          1. Strip whitespace from column names.
          2. Parse `date`; drop rows where date cannot be parsed.
          3. Coerce `Volume` to numeric; fill NaN → 0.
          4. Parse `updatetime`; fill missing with current timestamp.
          5. Fill missing `updatedby` with 'petroineos'.
          6. Map `country` from 2-letter codes to full names.
          7. Infer `Technology` (Gas / Wind) from the source filename.
          8. Deduplicate on (date, SiteName), keeping the row with the
             latest updatetime.
        """
        df = pd.read_csv(file_path)

        # 1.
        df.columns = df.columns.str.strip()
        df.columns = df.columns.str.lower()
        df = df.rename(columns={'sitename': 'SiteName', 'volume': 'Volume'})

        # Add missing columns if not present in source file
        if 'updatetime' not in df.columns:
            df['updatetime'] = pd.NaT
        if 'updatedby' not in df.columns:
            df['updatedby'] = pd.NA

        # 2.
        df['date'] = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')
        n_bad_dates = df['date'].isna().sum()
        if n_bad_dates:
            print(f'[analyse_plant_data] Dropping {n_bad_dates} row(s) with unparseable dates.')
        df = df.dropna(subset=['date']).reset_index(drop=True)

        # 3.
        df['Volume'] = pd.to_numeric(df['Volume'], errors='coerce')
        n_missing_vol = df['Volume'].isna().sum()
        if n_missing_vol:
            print(f'[analyse_plant_data] Filling {n_missing_vol} missing Volume value(s) with 0.')
        df['Volume'] = df['Volume'].fillna(0)

        # 4.
        df['updatetime'] = pd.to_datetime(df['updatetime'], errors='coerce')
        df['updatetime'] = df['updatetime'].fillna(pd.Timestamp.now())

        # 5.
        df['updatedby'] = df['updatedby'].fillna('petroineos')

        # 6.
        country_map = {
            'FR': 'France',
            'GB': 'Great Britain',
        }
        df['country'] = df['country'].map(
            lambda x: country_map.get(str(x).strip().upper(), str(x).strip()) if pd.notna(x) else 'Unknown'
        )

        # 7.
        technology = 'Wind' if 'wind' in os.path.basename(file_path).lower() else 'Gas'
        df['Technology'] = technology

        # 8.
        n_before = len(df)
        df = (
            df
            .sort_values('updatetime', ascending=False)
            .drop_duplicates(subset=['date', 'SiteName'], keep='first')
            .sort_values('date')
            .reset_index(drop=True)
        )
        n_dupes = n_before - len(df)
        if n_dupes:
            print(f'[analyse_plant_data] Removed {n_dupes} duplicate (date, SiteName) row(s).')

        db_cols = ['date', 'country', 'SiteName', 'Technology', 'updatedby', 'updatetime', 'Volume']
        return df[db_cols]

    def load_new_data_from_file(self, input_data: pd.DataFrame):
        db_cols = ['date', 'country', 'SiteName', 'Technology', 'updatedby', 'updatetime', 'Volume']
        missing = [c for c in db_cols if c not in input_data.columns]
        if missing:
            raise ValueError(f'Input DataFrame is missing required columns: {missing}')

        df = input_data.copy()

        # Normalisation
        df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')

        # Drop null SiteNames — cannot be keyed in the database
        n_before = len(df)
        df = df.dropna(subset=['SiteName']).reset_index(drop=True)
        if len(df) < n_before:
            print(f'[load_new_data_from_file] Dropped {n_before - len(df)} row(s) with null SiteName.')

        return df

    def save_new_data(self, input_data: pd.DataFrame):
        if not os.path.exists(self.database_file):
            input_data.to_csv(self.database_file, index=False)
            print(f'[save_new_data] Created {self.database_file} with {len(input_data)} rows.')
            return

        existing = pd.read_csv(self.database_file)

        # Normalise date strings
        existing['date'] = pd.to_datetime(existing['date']).dt.strftime('%Y-%m-%d')
        incoming = input_data.copy()
        incoming['date'] = pd.to_datetime(incoming['date']).dt.strftime('%Y-%m-%d')

        # Index tuples for upsert key matching
        existing_keys = set(zip(existing['date'], existing['SiteName']))
        incoming_keys = set(zip(incoming['date'], incoming['SiteName']))

        mask_keep = [
            (d, s) not in incoming_keys
            for d, s in zip(existing['date'], existing['SiteName'])
        ]
        kept = existing[mask_keep]

        combined = (
            pd.concat([kept, incoming], ignore_index=True)
            .sort_values(['date', 'SiteName'])
            .reset_index(drop=True)
        )
        combined.to_csv(self.database_file, index=False)

        n_updated = len(existing_keys & incoming_keys)
        n_new = len(incoming) - n_updated
        print(f'[save_new_data] Updated {n_updated} row(s), added {n_new} new row(s). '
              f'Database now has {len(combined)} rows.')

    def get_data_from_database(self):
        """
        Returns the most recently updated row for every (date, SiteName) in
        database.csv. If the same (date, SiteName) was uploaded more than once,
        only the row with the latest updatetime is kept — giving a full daily
        history with no duplicate entries per plant per day.
        """
        if not os.path.exists(self.database_file):
            print('[get_data_from_database] Database file not found.')
            return pd.DataFrame()

        db = pd.read_csv(self.database_file)
        db['updatetime'] = pd.to_datetime(db['updatetime'])

        latest = (
            db
            .sort_values('updatetime', ascending=False)
            .drop_duplicates(subset=['date', 'SiteName'], keep='first')
            .sort_values(['date', 'SiteName'])
            .reset_index(drop=True)
        )
        return latest

    def aggregate_data_to_quarterly(self):
        """
        Returns quarterly Mean, Median and Std of Volume per plant in wide format,
        matching the shape shown in month_average_min_mlx.png but at quarterly
        grain with sd/median as instructed by the spec comment.
        Quarters: Q1 Jan-Mar | Q2 Apr-Jun | Q3 Jul-Sep | Q4 Oct-Dec
        """
        if not os.path.exists(self.database_file):
            print('[aggregate_data_to_quarterly] Database file not found.')
            return pd.DataFrame()

        db = pd.read_csv(self.database_file)
        db['date'] = pd.to_datetime(db['date'])
        db['Volume'] = pd.to_numeric(db['Volume'], errors='coerce').fillna(0)

        # Quarter-start date as the grouping key
        db['quarter'] = db['date'].dt.to_period('Q').dt.to_timestamp()

        agg = (
            db
            .groupby(['quarter', 'SiteName'])['Volume']
            .agg(Mean='mean', Median='median', Std='std')
            .round(6)
        )

        # Pivot to wide: one column triplet per SiteName — mirrors the png structure
        wide = agg.unstack(level='SiteName')
        wide.columns = [f'{site} {stat}' for stat, site in wide.columns]
        wide = wide.reset_index().rename(columns={'quarter': 'date'})

        return wide

    def aggregate_data_to_country(self):
        """
        Returns total Volume by country and Technology, matching the structure
        shown in country_sum.png: MultiIndex (country, Technology) with Volume column.
        """
        if not os.path.exists(self.database_file):
            print('[aggregate_data_to_country] Database file not found.')
            return pd.DataFrame()

        db = pd.read_csv(self.database_file)
        db['Volume'] = pd.to_numeric(db['Volume'], errors='coerce').fillna(0)

        return (
            db
            .groupby(['country', 'Technology'])['Volume']
            .sum()
            .rename('Volume')
            .to_frame()
        )

In [89]:
pp = PowerPlants()
new_data = pp.load_new_data_from_file(pp.analyse_plant_data('wind_plants.csv'))
pp.save_new_data(new_data)
new_data = pp.load_new_data_from_file(pp.analyse_plant_data('gas_plants.csv'))
pp.save_new_data(new_data)
new_data = pp.load_new_data_from_file(pp.analyse_plant_data('gas_fr_plants.csv'))
pp.save_new_data(new_data)

[analyse_plant_data] Dropping 576 row(s) with unparseable dates.
[save_new_data] Updated 381 row(s), added 0 new row(s). Database now has 957 rows.
[analyse_plant_data] Dropping 578 row(s) with unparseable dates.
[save_new_data] Updated 384 row(s), added 0 new row(s). Database now has 957 rows.
[analyse_plant_data] Dropping 770 row(s) with unparseable dates.
[save_new_data] Updated 192 row(s), added 0 new row(s). Database now has 957 rows.


In [103]:
display(pp.get_data_from_database())

,date,country,SiteName,Technology,updatedby,updatetime,Volume
0,2024-01-01,France,Blenod-5,Gas,petroineos,2026-06-04 08:12:43.463330,6753.000000
1,2025-11-04,Great Britain,Hornsea-1,Wind,petroineos,2026-06-04 08:12:43.421902,524.540857
2,2025-04-01,Great Britain,Hornsea-2,Wind,petroineos,2026-06-04 08:12:43.421902,357.003005
3,2024-09-10,Great Britain,Pembroke-1,Gas,petroineos,2026-06-04 08:12:43.438941,6534.000000
4,2024-09-10,Great Britain,Pembroke-2,Gas,petroineos,2026-06-04 08:12:43.438941,5308.000000


In [91]:
display(pp.aggregate_data_to_quarterly())

,date,Blenod-5 Mean,Hornsea-1 Mean,Hornsea-2 Mean,Pembroke-1 Mean,Pembroke-2 Mean,Blenod-5 Median,Hornsea-1 Median,Hornsea-2 Median,Pembroke-1 Median,Pembroke-2 Median,Blenod-5 Std,Hornsea-1 Std,Hornsea-2 Std,Pembroke-1 Std,Pembroke-2 Std
0,2024-01-01,5098.305556,457.625534,463.800600,7215.666667,6936.916667,4990.0,449.422783,513.099616,7288.0,6682.5,1195.461150,238.431103,252.180124,1128.301631,1190.190414
1,2024-04-01,4844.944444,413.346528,377.702037,6956.055556,6724.166667,4605.0,408.503471,392.032833,7087.5,6422.0,1114.822983,284.919701,362.503595,1240.297567,1132.189648
2,2024-07-01,4885.194444,484.934702,463.835929,7265.805556,6949.555556,4800.0,472.108409,432.687978,7204.5,6911.0,1153.457159,297.085753,283.790410,1057.559423,1203.671347
3,2024-10-01,5083.500000,504.776557,570.123566,7229.222222,6901.194444,5276.0,569.394887,583.282855,7505.5,6791.5,1244.614398,291.434475,281.546293,1144.370722,1210.815895
4,2025-01-01,4590.416667,507.851307,603.601978,7088.166667,7528.666667,4166.5,471.297300,728.248827,6979.5,7638.0,985.638921,243.350500,279.522219,1004.097861,917.663172
5,2025-04-01,5210.583333,495.382855,516.153167,7006.166667,7172.916667,5534.5,480.423302,521.177765,7177.5,7413.0,1346.502031,297.133555,291.030912,1057.318292,1221.356210
6,2025-07-01,4534.333333,410.295486,424.797308,6986.500000,7393.500000,4265.0,370.157279,496.780436,7040.5,7737.5,1376.813933,265.564573,319.252737,1344.532125,1229.380738
7,2025-10-01,4709.833333,520.321264,439.519398,6669.000000,7354.333333,4190.0,513.909892,404.125274,6552.0,7644.0,1248.138893,246.079853,292.513347,829.228886,875.046735


In [92]:
display(pp.aggregate_data_to_country())

Volume
country       Technology              
France        Gas         9.453720e+05
Great Britain Gas         2.708820e+06
              Wind        1.798059e+05